# Reproducibility Manifest 2 (Unified & Exhaustive): Remaining Tasks
This notebook executes the remaining empirical tasks for ALL models (Pointwise and Sequence) on a GPU.

In [ ]:
import os, sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
SUPERVISED_DIR = str(PROJECT_ROOT)
if SUPERVISED_DIR not in sys.path:
    sys.path.append(SUPERVISED_DIR)
os.chdir(SUPERVISED_DIR)

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from implement.utils.dataset_processing.dataset_helper import combine_and_split_supervised_data, impute_and_scale_data
from implement.utils.helper import get_or_preprocess_genuine_dji_flights, get_genuine_dji_flights_with_device_split, get_or_preprocess_hardware_spoofer, SUPERVISED_FEATURES
from implement.utils.classical_ml.classical_models import get_point_classifiers
from sklearn.metrics import recall_score, f1_score
from implement.utils.helper.ablation_utils import apply_ablation_overrides
from sklearn.model_selection import GroupKFold
from implement.utils.deep_learning.sequence_helper import generate_sequences_from_df
from implement.utils.deep_learning import train_cnn_classifier, train_gru_classifier, train_tcn_classifier, train_cnn_gru_classifier, evaluate_model
import implement.utils.helper.features as feat

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)

CLASSICAL_MODELS = ["Logistic Regression", "Random Forest", "Threshold(PE)", "XGBoost"]
SEQUENCE_MODELS = ["CNN", "GRU", "CNN-GRU", "TCN"]

## 1a. Per-Class Recall (Device Split)
Evaluating how well ALL models detect specific sub-classes under cross-hardware testing. **Note:** We use the device split here because the reviewer explicitly asked to explain the 100% precision anomaly that occurred specifically in Table 4 (the Device-Wise split table).

In [ ]:

print("=== 1a. Per-Class Recall (Device Split) ===")
dji_df = get_genuine_dji_flights_with_device_split(filter_length_100=True)
esp32_df = get_or_preprocess_hardware_spoofer()

# 1. Pointwise Prep & Train
X_train_pt, y_train_pt, X_test_pt, y_test_pt, scaler, imputer = combine_and_split_supervised_data(
    dji_df, esp32_df, train_ratio=0.8, split_mode='device', random_state=42, features=SUPERVISED_FEATURES
)
pt_models = get_point_classifiers(random_state=42)
trained_pt_models = {}
for name in CLASSICAL_MODELS:
    pt_models[name].fit(X_train_pt, y_train_pt)
    trained_pt_models[name] = pt_models[name]

# 2. Sequence Prep & Train
print("\nPreparing sequence splits...")
if 'device' == 'device':
    dji_train = dji_df[~dji_df['drone_model'].isin(['dji_phantom_3', 'dji_inspire_2', 'dji_mavic_2'])]
else:
    # Basic flight split emulation (approx 80/20 on unique flight IDs)
    flights = dji_df['flight_id'].unique()
    np.random.seed(42)
    shuf_flights = np.random.permutation(flights)
    train_f = shuf_flights[:int(0.8*len(flights))]
    dji_train = dji_df[dji_df['flight_id'].isin(train_f)]

esp32_train = esp32_df.sample(frac=0.8, random_state=42)

def prep_seqs(df_dji, df_esp):
    dji_seqs = [generate_sequences_from_df(df_dji[df_dji['flight_id'] == f], SUPERVISED_FEATURES, feat.WINDOW_LEN, step=1) for f in df_dji['flight_id'].unique()]
    esp_seqs = [generate_sequences_from_df(df_esp[df_esp['flight_id'] == f], SUPERVISED_FEATURES, feat.WINDOW_LEN, step=1) for f in df_esp['flight_id'].unique()]
    
    X_dji = np.concatenate(dji_seqs, axis=0) if dji_seqs else np.empty((0, feat.WINDOW_LEN, len(SUPERVISED_FEATURES)))
    X_esp = np.concatenate(esp_seqs, axis=0) if esp_seqs else np.empty((0, feat.WINDOW_LEN, len(SUPERVISED_FEATURES)))
    
    X = np.concatenate([X_dji, X_esp], axis=0)
    y = np.concatenate([np.zeros(len(X_dji)), np.ones(len(X_esp))])
    return X, y

X_tr_seq_raw, y_tr_seq = prep_seqs(dji_train, esp32_train)
seq_scaler = StandardScaler()
n_tr, t, f = X_tr_seq_raw.shape
X_tr_seq = seq_scaler.fit_transform(X_tr_seq_raw.reshape(-1, f)).reshape(n_tr, t, f)
X_tr_seq = np.nan_to_num(X_tr_seq, nan=0.0)

shuf = np.random.permutation(len(X_tr_seq))
X_tr_seq, y_tr_seq = X_tr_seq[shuf], y_tr_seq[shuf]

trained_seq_models = {}
print("Training CNN...")
trained_seq_models['CNN'], _ = train_cnn_classifier(X_tr_seq, y_tr_seq, X_tr_seq[:100], y_tr_seq[:100], epochs=15, batch_size=64, device=device, hidden_dim=64, tune_threshold=False)
print("Training GRU...")
trained_seq_models['GRU'], _ = train_gru_classifier(X_tr_seq, y_tr_seq, X_tr_seq[:100], y_tr_seq[:100], epochs=15, batch_size=64, device=device, hidden_dim=64, num_layers=2, tune_threshold=False)
print("Training TCN...")
trained_seq_models['TCN'], _ = train_tcn_classifier(X_tr_seq, y_tr_seq, X_tr_seq[:100], y_tr_seq[:100], epochs=15, batch_size=64, device=device, hidden_dim=64, tune_threshold=False)
print("Training CNN-GRU...")
trained_seq_models['CNN-GRU'], _ = train_cnn_gru_classifier(X_tr_seq, y_tr_seq, X_tr_seq[:100], y_tr_seq[:100], epochs=15, batch_size=64, device=device, hidden_dim=64, num_layers=2, tune_threshold=False)

print("\nEvaluating Recall per Attack Class:")
esp32_real = esp32_df[esp32_df['flight_id'] == 'esp32_flight'].sample(n=min(1500, len(esp32_df[esp32_df['flight_id'] == 'esp32_flight'])), random_state=42)
def eval_class(name, df):
    if len(df) == 0: return
    # Eval Pointwise
    X_pt = scaler.transform(imputer.transform(df[SUPERVISED_FEATURES]))
    pt_recalls = []
    for m_name in CLASSICAL_MODELS:
        pt_preds = trained_pt_models[m_name].predict(X_pt)
        pt_recalls.append(f"{pt_preds.mean():.2%}")
    
    # Eval Sequence
    X_seq_raw, y_seq = prep_seqs(pd.DataFrame(columns=dji_df.columns), df)
    n, t, f_dims = X_seq_raw.shape
    seq_recalls = []
    if n > 0:
        X_seq = seq_scaler.transform(X_seq_raw.reshape(-1, f_dims)).reshape(n, t, f_dims)
        X_seq = np.nan_to_num(X_seq, nan=0.0)
        X_tensor = torch.FloatTensor(X_seq).to(device)
        for m_name in SEQUENCE_MODELS:
            model = trained_seq_models[m_name]
            model.eval()
            with torch.no_grad():
                preds = (model(X_tensor).squeeze().cpu().numpy() > 0.5).astype(int)
                if preds.ndim == 0:
                    preds = np.array([preds])
                seq_recalls.append(f"{recall_score(y_seq, preds, zero_division=0):.2%}")
    else:
        seq_recalls = ["0.00%"] * len(SEQUENCE_MODELS)
        
    print(f"{name:15} | PT: {', '.join(pt_recalls)} | SEQ: {', '.join(seq_recalls)}")

print("Format: Class | PT: [LR, RF, Threshold(PE), XGB] | SEQ: [CNN, GRU, CNN-GRU, TCN]")
eval_class("Real ESP32", esp32_real)
sim_flights = [f for f in esp32_df['flight_id'].unique() if f != 'esp32_flight']
sim_groups = {}
for f in sim_flights:
    sim_groups.setdefault(f.split('_flight_')[0] if '_flight_' in f else f.split('_')[0], []).append(f)

for cls, flights in sim_groups.items():
    eval_class(f"Class {cls}", esp32_df[esp32_df['flight_id'].isin(flights)])

## 1b. Per-Class Recall (Flight Split)
Out of curiosity, evaluating per-class recall under standard Flight-Wise splitting to see how it differs from Device splitting.

In [ ]:

print("=== 1b. Per-Class Recall (Flight Split) ===")
dji_df = get_genuine_dji_flights_with_device_split(filter_length_100=True)
esp32_df = get_or_preprocess_hardware_spoofer()

# 1. Pointwise Prep & Train
X_train_pt, y_train_pt, X_test_pt, y_test_pt, scaler, imputer = combine_and_split_supervised_data(
    dji_df, esp32_df, train_ratio=0.8, split_mode='flight', random_state=42, features=SUPERVISED_FEATURES
)
pt_models = get_point_classifiers(random_state=42)
trained_pt_models = {}
for name in CLASSICAL_MODELS:
    pt_models[name].fit(X_train_pt, y_train_pt)
    trained_pt_models[name] = pt_models[name]

# 2. Sequence Prep & Train
print("\nPreparing sequence splits...")
if 'flight' == 'device':
    dji_train = dji_df[~dji_df['drone_model'].isin(['dji_phantom_3', 'dji_inspire_2', 'dji_mavic_2'])]
else:
    # Basic flight split emulation (approx 80/20 on unique flight IDs)
    flights = dji_df['flight_id'].unique()
    np.random.seed(42)
    shuf_flights = np.random.permutation(flights)
    train_f = shuf_flights[:int(0.8*len(flights))]
    dji_train = dji_df[dji_df['flight_id'].isin(train_f)]

esp32_train = esp32_df.sample(frac=0.8, random_state=42)

def prep_seqs(df_dji, df_esp):
    dji_seqs = [generate_sequences_from_df(df_dji[df_dji['flight_id'] == f], SUPERVISED_FEATURES, feat.WINDOW_LEN, step=1) for f in df_dji['flight_id'].unique()]
    esp_seqs = [generate_sequences_from_df(df_esp[df_esp['flight_id'] == f], SUPERVISED_FEATURES, feat.WINDOW_LEN, step=1) for f in df_esp['flight_id'].unique()]
    
    X_dji = np.concatenate(dji_seqs, axis=0) if dji_seqs else np.empty((0, feat.WINDOW_LEN, len(SUPERVISED_FEATURES)))
    X_esp = np.concatenate(esp_seqs, axis=0) if esp_seqs else np.empty((0, feat.WINDOW_LEN, len(SUPERVISED_FEATURES)))
    
    X = np.concatenate([X_dji, X_esp], axis=0)
    y = np.concatenate([np.zeros(len(X_dji)), np.ones(len(X_esp))])
    return X, y

X_tr_seq_raw, y_tr_seq = prep_seqs(dji_train, esp32_train)
seq_scaler = StandardScaler()
n_tr, t, f = X_tr_seq_raw.shape
X_tr_seq = seq_scaler.fit_transform(X_tr_seq_raw.reshape(-1, f)).reshape(n_tr, t, f)
X_tr_seq = np.nan_to_num(X_tr_seq, nan=0.0)

shuf = np.random.permutation(len(X_tr_seq))
X_tr_seq, y_tr_seq = X_tr_seq[shuf], y_tr_seq[shuf]

trained_seq_models = {}
print("Training CNN...")
trained_seq_models['CNN'], _ = train_cnn_classifier(X_tr_seq, y_tr_seq, X_tr_seq[:100], y_tr_seq[:100], epochs=15, batch_size=64, device=device, hidden_dim=64, tune_threshold=False)
print("Training GRU...")
trained_seq_models['GRU'], _ = train_gru_classifier(X_tr_seq, y_tr_seq, X_tr_seq[:100], y_tr_seq[:100], epochs=15, batch_size=64, device=device, hidden_dim=64, num_layers=2, tune_threshold=False)
print("Training TCN...")
trained_seq_models['TCN'], _ = train_tcn_classifier(X_tr_seq, y_tr_seq, X_tr_seq[:100], y_tr_seq[:100], epochs=15, batch_size=64, device=device, hidden_dim=64, tune_threshold=False)
print("Training CNN-GRU...")
trained_seq_models['CNN-GRU'], _ = train_cnn_gru_classifier(X_tr_seq, y_tr_seq, X_tr_seq[:100], y_tr_seq[:100], epochs=15, batch_size=64, device=device, hidden_dim=64, num_layers=2, tune_threshold=False)

print("\nEvaluating Recall per Attack Class:")
esp32_real = esp32_df[esp32_df['flight_id'] == 'esp32_flight'].sample(n=min(1500, len(esp32_df[esp32_df['flight_id'] == 'esp32_flight'])), random_state=42)
def eval_class(name, df):
    if len(df) == 0: return
    # Eval Pointwise
    X_pt = scaler.transform(imputer.transform(df[SUPERVISED_FEATURES]))
    pt_recalls = []
    for m_name in CLASSICAL_MODELS:
        pt_preds = trained_pt_models[m_name].predict(X_pt)
        pt_recalls.append(f"{pt_preds.mean():.2%}")
    
    # Eval Sequence
    X_seq_raw, y_seq = prep_seqs(pd.DataFrame(columns=dji_df.columns), df)
    n, t, f_dims = X_seq_raw.shape
    seq_recalls = []
    if n > 0:
        X_seq = seq_scaler.transform(X_seq_raw.reshape(-1, f_dims)).reshape(n, t, f_dims)
        X_seq = np.nan_to_num(X_seq, nan=0.0)
        X_tensor = torch.FloatTensor(X_seq).to(device)
        for m_name in SEQUENCE_MODELS:
            model = trained_seq_models[m_name]
            model.eval()
            with torch.no_grad():
                preds = (model(X_tensor).squeeze().cpu().numpy() > 0.5).astype(int)
                if preds.ndim == 0:
                    preds = np.array([preds])
                seq_recalls.append(f"{recall_score(y_seq, preds, zero_division=0):.2%}")
    else:
        seq_recalls = ["0.00%"] * len(SEQUENCE_MODELS)
        
    print(f"{name:15} | PT: {', '.join(pt_recalls)} | SEQ: {', '.join(seq_recalls)}")

print("Format: Class | PT: [LR, RF, Threshold(PE), XGB] | SEQ: [CNN, GRU, CNN-GRU, TCN]")
eval_class("Real ESP32", esp32_real)
sim_flights = [f for f in esp32_df['flight_id'].unique() if f != 'esp32_flight']
sim_groups = {}
for f in sim_flights:
    sim_groups.setdefault(f.split('_flight_')[0] if '_flight_' in f else f.split('_')[0], []).append(f)

for cls, flights in sim_groups.items():
    eval_class(f"Class {cls}", esp32_df[esp32_df['flight_id'].isin(flights)])

## 2. ESP32 Ablation (Without motion_smoothness)
Excluding the resampling artifact to ensure models detect physical anomalies. Running ALL models.

In [ ]:
from pathlib import Path

print("\n=== ESP32 Ablation (All Models) ===")
from implement.workflows.leave_one_attack_out_unified import run_leave_one_attack_out_workflow
apply_ablation_overrides(None, "motion_smoothness", Path(SUPERVISED_DIR))
run_leave_one_attack_out_workflow(model_type="all", random_state=42, epochs=15, batch_size=64, output_suffix="ablation_ms_unified")
apply_ablation_overrides(None, None, Path(SUPERVISED_DIR))

## 3. Device-Wise 5-Fold Cross Validation (Unified)
Executing 5-fold CV holding out drone models across ALL models.

In [ ]:

print("\n=== Device-Wise 5-Fold CV (All Models) ===")
dji_df = get_genuine_dji_flights_with_device_split(filter_length_100=True)
esp32_df = get_or_preprocess_hardware_spoofer()

unique_models = dji_df['drone_model'].unique()
np.random.seed(42)
shuffled_models = np.random.permutation(unique_models)
folds = np.array_split(shuffled_models, 5)

cv_f1s = {m: [] for m in CLASSICAL_MODELS + SEQUENCE_MODELS}

for i, test_models in enumerate(folds):
    print(f"\n--- Fold {i+1} ---")
    train_models = np.setdiff1d(unique_models, test_models)
    
    dji_train = dji_df[dji_df['drone_model'].isin(train_models)]
    dji_test = dji_df[dji_df['drone_model'].isin(test_models)]
    
    esp32_train = esp32_df.sample(frac=0.8, random_state=42+i)
    esp32_test = esp32_df.drop(esp32_train.index)
    
    # 1. Pointwise Pipeline
    train_df = pd.concat([dji_train.drop(columns=['drone_model', 'flight_id'], errors='ignore'), esp32_train.drop(columns=['flight_id'], errors='ignore')])
    test_df = pd.concat([dji_test.drop(columns=['drone_model', 'flight_id'], errors='ignore'), esp32_test.drop(columns=['flight_id'], errors='ignore')])
    
    X_tr_pt, X_te_pt, scaler, imputer = impute_and_scale_data(train_df[SUPERVISED_FEATURES], test_df[SUPERVISED_FEATURES])
    y_tr_pt, y_te_pt = train_df['nature'].to_numpy(), test_df['nature'].to_numpy()
    
    pt_models = get_point_classifiers(random_state=42)
    for name in CLASSICAL_MODELS:
        pt_models[name].fit(X_tr_pt, y_tr_pt)
        cv_f1s[name].append(f1_score(y_te_pt, pt_models[name].predict(X_te_pt), zero_division=0))
    
    # 2. Sequence Pipeline
    def prep_seqs(df_dji, df_esp):
        dji_seqs = [generate_sequences_from_df(df_dji[df_dji['flight_id'] == f], SUPERVISED_FEATURES, feat.WINDOW_LEN, step=1) for f in df_dji['flight_id'].unique()]
        esp_seqs = [generate_sequences_from_df(df_esp[df_esp['flight_id'] == f], SUPERVISED_FEATURES, feat.WINDOW_LEN, step=1) for f in df_esp['flight_id'].unique()]
        X_dji = np.concatenate(dji_seqs, axis=0) if dji_seqs else np.empty((0, feat.WINDOW_LEN, len(SUPERVISED_FEATURES)))
        X_esp = np.concatenate(esp_seqs, axis=0) if esp_seqs else np.empty((0, feat.WINDOW_LEN, len(SUPERVISED_FEATURES)))
        X = np.concatenate([X_dji, X_esp], axis=0)
        y = np.concatenate([np.zeros(len(X_dji)), np.ones(len(X_esp))])
        return X, y
        
    X_tr_seq_raw, y_tr_seq = prep_seqs(dji_train, esp32_train)
    X_te_seq_raw, y_te_seq = prep_seqs(dji_test, esp32_test)
    
    seq_scaler = StandardScaler()
    n_tr, t, f = X_tr_seq_raw.shape
    n_te = X_te_seq_raw.shape[0]
    
    X_tr_seq = seq_scaler.fit_transform(X_tr_seq_raw.reshape(-1, f)).reshape(n_tr, t, f)
    X_te_seq = seq_scaler.transform(X_te_seq_raw.reshape(-1, f)).reshape(n_te, t, f)
    X_tr_seq = np.nan_to_num(X_tr_seq, nan=0.0)
    X_te_seq = np.nan_to_num(X_te_seq, nan=0.0)
    
    shuf = np.random.permutation(len(X_tr_seq))
    X_tr_seq, y_tr_seq = X_tr_seq[shuf], y_tr_seq[shuf]
    
    cnn, _ = train_cnn_classifier(X_tr_seq, y_tr_seq, X_te_seq, y_te_seq, epochs=15, batch_size=64, device=device, hidden_dim=64, tune_threshold=False)
    cv_f1s['CNN'].append(evaluate_model(cnn, X_te_seq, y_te_seq, batch_size=64, device=device)['f1_score'])
    
    gru, _ = train_gru_classifier(X_tr_seq, y_tr_seq, X_te_seq, y_te_seq, epochs=15, batch_size=64, device=device, hidden_dim=64, num_layers=2, tune_threshold=False)
    cv_f1s['GRU'].append(evaluate_model(gru, X_te_seq, y_te_seq, batch_size=64, device=device)['f1_score'])

    tcn, _ = train_tcn_classifier(X_tr_seq, y_tr_seq, X_te_seq, y_te_seq, epochs=15, batch_size=64, device=device, hidden_dim=64, tune_threshold=False)
    cv_f1s['TCN'].append(evaluate_model(tcn, X_te_seq, y_te_seq, batch_size=64, device=device)['f1_score'])

    cg, _ = train_cnn_gru_classifier(X_tr_seq, y_tr_seq, X_te_seq, y_te_seq, epochs=15, batch_size=64, device=device, hidden_dim=64, num_layers=2, tune_threshold=False)
    cv_f1s['CNN-GRU'].append(evaluate_model(cg, X_te_seq, y_te_seq, batch_size=64, device=device)['f1_score'])
    
    print(f"Test Models: {list(test_models)}")
    for m in CLASSICAL_MODELS + SEQUENCE_MODELS:
        print(f"  {m:20} F1: {cv_f1s[m][-1]:.2%}")

print(f"\n--- Final Device-Wise CV Results ---")
for m in CLASSICAL_MODELS + SEQUENCE_MODELS:
    print(f"{m:20} F1: {np.mean(cv_f1s[m]):.2%} ± {np.std(cv_f1s[m]):.2%}")